In [181]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

For an object with mass $m$ and moment of inertia matrix $\mathrm{I}_p$ about a pivot which has velocity $\mathbf{v}_p$, where the displacement from the pivot to the center of mass is $\mathbf{r}_{\text{pivot} \rightarrow \text{com}}$, and the object is rotating with angular velocity $\omega$ about the pivot, the kinetic energy is
$$
T = \frac{1}{2} m \mathbf{v}^2_p + m \mathbf{v}_p \cdot (\mathbf{\omega} \times \mathbf{r}_{\text{pivot} \rightarrow \text{com}}) + \frac{1}{2}\mathbf{\omega} \cdot \mathrm{I}_p \mathbf{\omega} .
$$

We set $\omega = \dot{\theta} \hat{k}$, so the only important component of moment of inertia is $I = \mathrm{I}_{p, 3,3}$. We also get $\mathbf{r}_{\text{pivot} \rightarrow \text{com}} = \dot\theta l \hat\theta$ where $l$ is the distance from the pivot to the center of mass. We also have the pivot's velocity only in the $\hat{j}$ direction and we call its magnitude $\dot{d}$.

We therefore have
$$
T = \frac{1}{2} m \dot{d}^2 + ml\dot{d}\dot\theta \sin\theta + \frac{1}{2}I\dot{\theta}^2 .
$$

For potential energy, we get
$$
V = mgy = mg(d-l\cos\theta) .
$$

Plugging these into the Euler-Lagrange equation, we get
$$
\ddot\theta + \frac{ml}{I} (g-\ddot{d})\sin\theta = 0 .
$$
Let's define $\lambda = \frac{I}{ml}$ as the effective length of the pendulum. It acts like the length does in a simple pendulum with a point mass on a massless rod.

In [182]:
def angular_acceleration(eff_length, gravity, pivot_acc, theta):
    return -np.sin(theta) * (gravity - pivot_acc) / eff_length

def angular_acceleration_gravity(eff_length, gravity, theta):
    return -np.sin(theta) * gravity / eff_length

def angular_acceleration_pivot(eff_length, pivot_acc, theta):
    return np.sin(theta) * pivot_acc / eff_length

def pivot_acceleration(amplitude, frequency, t):
    return -amplitude * frequency * frequency * np.cos(frequency * t)

In [183]:
def yoshida_4th_step(q, p, t, dt, get_force, get_velocity):
    """General 4th-Order Yoshida symplectic integrator step."""
    w1 = 1.0 / (2.0 - 2.0 ** (1.0 / 3.0))
    w0 = -(2.0 ** (1.0 / 3.0)) / (2.0 - 2.0 ** (1.0 / 3.0))

    d = [w1, w0, w1, 0.0]

    curr_q, curr_p = q, p
    curr_t = t

    for i in range(4):
        dt_sub = d[i] * dt
        p_half = curr_p + 0.5 * dt_sub * get_force(curr_q, curr_t)
        curr_q = curr_q + dt_sub * get_velocity(p_half)
        curr_p = p_half + 0.5 * dt_sub * get_force(curr_q, curr_t + dt_sub)
        curr_t += dt_sub

    return curr_q, curr_p

In [184]:
g = 9.81
l = 1.0
m = 1.0
I = m * (l**2)
eff_length = I / m / l
A = 0.05
omega = 150.0 # this is set to be just enough to keep it up

In [185]:
dt = np.min([1 / np.float64(100.0) / omega, 0.001])
t_max = 2.0
steps = int(t_max / dt)
t_steps = np.arange(0, t_max, dt)

In [186]:
def pivot_position(A, omega, t):
    return A * np.cos(omega * t)


def kapitza_velocity(p):
    return p / I # angular momentum divided by moment of inertia


def kapitza_force(theta, t):
    return I * angular_acceleration(
        eff_length, g, pivot_acceleration(A, omega, t), theta
    )

In [187]:
theta_init = np.pi - 0.1
p_init = 0.0

In [189]:
th, p = theta_init, p_init

theta_vals = np.zeros_like(t_steps)
gravity_acc_mag = np.zeros_like(t_steps)
pivot_acc_mag = np.zeros_like(t_steps)

print('Running simulation...')
for i, t in enumerate(t_steps):
    print(i)
    theta_vals[i] = th
    th, p = yoshida_4th_step(th, p, t, dt, kapitza_force, kapitza_velocity)

    gravity_acc_mag[i] = angular_acceleration_gravity(eff_length, g, th)
    pivot_acc_mag[i] = angular_acceleration_pivot(eff_length, pivot_acceleration(A, omega, t), th)

print('Simulation complete. Generating animation...')


# Animation Setup for Jupyter with Tangential Acceleration Vectors
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True)
ax.set_title(
    'Kapitza Pendulum: Tangential Accelerations\n(Red: Gravity Torque, Green:'
    ' Pivot Torque)'
)

(line,) = ax.plot([], [], 'o-', lw=3, markersize=10, color='royalblue')
time_text = ax.text(
    0.05, 0.88, '', transform=ax.transAxes, fontsize=11, fontweight='bold'
)

# Quiver arrows for tangential components
quiver_gravity = ax.quiver(
    0,
    0,
    0,
    0,
    angles='xy',
    scale_units='xy',
    scale=1,
    color='red',
    label='Gravity Tangential Accel',
)
quiver_pivot = ax.quiver(
    0,
    0,
    0,
    0,
    angles='xy',
    scale_units='xy',
    scale=1,
    color='green',
    label='Pivot Tangential Accel',
)
ax.legend(loc='upper right')

# Skip frames to keep playback smooth
frame_skip = max(1, int(0.01 / dt))
frames_to_animate = range(0, len(t_steps), frame_skip)


def init():
    line.set_data([], [])
    time_text.set_text('')
    quiver_gravity.set_UVC(0, 0)
    quiver_pivot.set_UVC(0, 0)
    return line, time_text, quiver_gravity, quiver_pivot


def animate(i):
    t = t_steps[i]
    theta = theta_vals[i]

    # Pivot coordinates (moves vertically)
    x_pivot = 0.0
    # y_pivot = pivot_position(A, omega, t)
    y_pivot = 0.0

    # Bob coordinates
    x_bob = x_pivot + l * np.sin(theta)
    y_bob = y_pivot - l * np.cos(theta)

    line.set_data([x_pivot, x_bob], [y_pivot, y_bob])

    # Tangential unit vector components: u_tan = (cos(theta), sin(theta))
    u_tan_x = np.cos(theta)
    u_tan_y = np.sin(theta)

    # Scale accelerations for visual representation in the plot space
    scale_factor = 10 / (A * omega**2)
    g_mag = gravity_acc_mag[i] * scale_factor
    p_mag = pivot_acc_mag[i] * scale_factor

    # Update gravity tangential arrow (at the bob)
    quiver_gravity.set_offsets([x_bob, y_bob])
    quiver_gravity.set_UVC(g_mag * u_tan_x, g_mag * u_tan_y)

    # Update pivot tangential arrow (offset slightly for clarity)
    quiver_pivot.set_offsets([x_bob, y_bob + 0.05])
    quiver_pivot.set_UVC(p_mag * u_tan_x, p_mag * u_tan_y)

    time_text.set_text(
        f'Time: {t:.2f} s\nGravity Tangent Accel: {gravity_acc_mag[i]:.1f}\nPivot'
        f' Tangent Accel: {pivot_acc_mag[i]:.1f}'
    )
    return line, time_text, quiver_gravity, quiver_pivot


ani = animation.FuncAnimation(
    fig,
    animate,
    init_func=init,
    frames=frames_to_animate,
    interval=20,
    blit=False,
)

# writer = animation.PillowWriter(fps=50)
# ani.save('kapitza_pendulum.gif', writer=writer)

plt.close(fig)
HTML(ani.to_jshtml())

Running simulation...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
